# 04. GGUF 양자화 — llama.cpp + Langfuse 품질 손실 측정


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)을 소재로 합니다.

| 항목 | 내용 |
|------|------|
| 목적 | 02 SFT FP16 모델을 GGUF(FP16/Q8_0/Q4_K_M) 3종으로 양자화하고 품질 손실 정량 측정 |
| 원본 모델 | 02에서 HF Hub에 배포한 `<user>/product-cs-qwen3.5-2b-sft` (FP16 merged) |
| 추론 엔진 | `llama-cpp-python` (CPU 추론) |
| 평가 질문 | 03과 동일한 30개 (`random.seed(42)`) |
| 심사위원 | OpenAI GPT-4o-mini |
| 모니터링 | Langfuse Cloud (**03과 동일 Project 재사용**) |
| 런타임 | **CPU** (Colab 무료 CPU 런타임 · 맥북 · 로컬 노트북 모두 가능, GPU 불필요) |
| 예상 소요 | ~40~55분 |
| 예상 비용 | ~$0.3 (90개 답변 평가) |

> **사전 조건**
> 1. `02_sft_train.ipynb`에서 **FP16 merged**로 `<user>/product-cs-qwen3.5-2b-sft` 배포 완료 (`save_method="merged_16bit"`)
> 2. `03_sft_eval.ipynb`에서 **SFT+NoPrompt (FP16)** Score가 Langfuse에 기록되어 있을 것 (양자화 손실 해석의 기준점)
> 3. `test.jsonl` (01 결과물)이 Drive 또는 로컬에 존재

---

### 왜 이 실습을 하는가?

14강에서 개념으로 배운 GGUF/Q4_K_M이 **실제 `.gguf` 파일**로 만들어지는 과정을 직접 수행하고, 같은 모델이 정밀도에 따라 **파일 크기가 3배 이상 차이** 나는 것과 **GPU 없이 CPU에서 그대로 돌아가는 것**을 눈으로 확인합니다.

**비교 프레임워크 (3조건, 시스템 프롬프트 없음)**:
- `SFT+FP16-gguf` → **포맷만** GGUF로 바꾼 baseline (원본 품질)
- `SFT+Q8_0` → 8bit 양자화 (절반 크기)
- `SFT+Q4_K_M` → 4bit 양자화 (1/3 크기, 업계 표준)

**해석 포인트**:
- `FP16-gguf` ≈ 03의 `SFT+NoPrompt` 점수 → **포맷 변환은 품질 손실 없음** 확인
- `Q4_K_M` − `FP16-gguf` → **양자화 고유 손실** (목표: 5% 이내, 일반 답변 태스크)
- 수학/추론 태스크(Phase 13)에서는 손실이 더 크게 나타날 수 있음 (14강 "오차 누적")


---
## Phase 1: 환경 설정 — llama.cpp 빌드 + 인증

### `llama.cpp`가 뭔가요?

**Georgi Gerganov**가 만든 **순수 C/C++ LLM 추론 엔진**입니다 (repo: [ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)). 원래는 "Meta의 LLaMA 모델을 맥북 CPU에서 돌려보자"는 개인 프로젝트로 시작됐지만, 지금은 **로컬/엣지 LLM 생태계의 사실상 표준 런타임**이 됐습니다.

- **순수 C/C++ + SIMD 최적화** → Python/CUDA 런타임 없이 바이너리 하나로 동작
- **GGUF 포맷 정의** → 양자화된 가중치 + tokenizer + 메타데이터를 한 파일에 패킹 (14강)
- **CPU 기본, GPU는 옵션** → 맥북 Apple Silicon, x86 CPU, 라즈베리파이까지 지원
- **생태계 허브 역할** → **Ollama, LM Studio** 등 인기 로컬 도구들이 내부적으로 llama.cpp를 엔진으로 사용합니다

이번 실습에서 우리가 쓰는 세 가지 결과물:
1. `convert_hf_to_gguf.py` — HF safetensors → GGUF 변환기 (Phase 3)
2. `llama-quantize` — FP16 GGUF → 저비트 GGUF 양자화기 (Phase 4)
3. `llama-cpp-python` — llama.cpp의 파이썬 바인딩, CPU 추론용 (Phase 7)

---

### 이 셀에서 할 일

- **런타임**: CPU (Colab 무료 CPU · 맥북 · 로컬 노트북 모두 OK, GPU 불필요)
- `llama.cpp`를 clone + cmake 빌드 → **`llama-quantize` 바이너리**만 만듭니다 (CPU용, CUDA 불필요)
- `convert_hf_to_gguf.py`가 쓰는 파이썬 의존성 설치
- 03과 동일한 4가지 토큰(HF · OpenAI · Langfuse Public · Secret) 입력


In [1]:
# llama.cpp 소스 clone + llama-quantize 타겟 빌드 (CPU-only)
#
# ⚠️ 버전 호환성:
#    llama-cpp-python 0.3.21 (Phase 7에서 사용)이 번들한 llama.cpp 커밋과 동일한 commit을
#    명시적으로 체크아웃합니다. HEAD를 받으면 convert_hf_to_gguf.py가 더 최신 GGUF 메타데이터를
#    생성해서 0.3.21의 reader가 "Failed to load model from file"로 거부합니다.
#
# ⚠️ Colab 무료 런타임 RAM(~13GB)에서는 기본 병렬 빌드(-j)가 cc1plus를 OOM으로 죽입니다.
#    각 C++ 파일 컴파일이 1~2GB를 쓰는데 여러 개가 동시에 돌면 메모리가 바닥나기 때문입니다.
#    -j 1(직렬 빌드)로 피해갑니다. 소요 시간: ~5~7분, 메모리 피크 ~2GB로 안정.

import os, subprocess
from pathlib import Path

# llama-cpp-python v0.3.21 vendored commit (PyPI 2026-04-27 release)
LLAMA_CPP_COMMIT = "f53577432541bb9edc1588c4ef45c66bf07e4468"

def current_commit():
    if not Path("llama.cpp/.git").exists():
        return None
    try:
        return subprocess.check_output(
            ["git", "-C", "llama.cpp", "rev-parse", "HEAD"],
            text=True,
        ).strip()
    except Exception:
        return None

existing = current_commit()
if existing != LLAMA_CPP_COMMIT:
    if existing:
        print(f"⚠️  기존 llama.cpp 커밋 {existing[:12]} → 정리하고 호환 커밋으로 재설치")
        !rm -rf llama.cpp gguf
    !git clone https://github.com/ggerganov/llama.cpp
    !cd llama.cpp && git checkout {LLAMA_CPP_COMMIT}
    print(f"✅ llama.cpp pinned to {LLAMA_CPP_COMMIT[:12]} (llama-cpp-python 0.3.21 호환)")
else:
    print(f"↩️  llama.cpp 이미 호환 커밋 {existing[:12]}")

# 빌드 (이미 있으면 재사용)
if not os.path.isfile("llama.cpp/build/bin/llama-quantize"):
    !cd llama.cpp && cmake -B build \
        -DLLAMA_CURL=OFF \
        -DGGML_NATIVE=OFF \
        -DCMAKE_BUILD_TYPE=Release > /dev/null
    !cd llama.cpp && cmake --build build --config Release -j 1 --target llama-quantize
else:
    print("↩️  기존 빌드된 llama-quantize 재사용")

# 파이썬 의존성 — llama.cpp/requirements.txt를 그대로 쓰면 protobuf를 4.25로 끌어내려서
# Colab 사전 설치 패키지들과 충돌 경고가 쏟아집니다. 실제로 필요한 건 `gguf` 뿐이고
# torch/transformers/sentencepiece/numpy는 Colab에 이미 있으므로 명시적으로 설치합니다.
!pip install --quiet gguf huggingface_hub openai langfuse tqdm matplotlib

# 💡 pip가 "ERROR: dependency resolver..." 경고를 띄울 수 있으나, Colab 사전 설치된
#    tensorflow/grain/ydf 등이 protobuf 버전 범위를 제각기 요구해서 생기는 경고일 뿐.
#    우리 코드 경로(convert_hf_to_gguf.py, llama-cpp-python, langfuse, openai)는
#    최종 protobuf 6.x와 호환되므로 무시해도 됩니다.

print("\n✅ llama.cpp 빌드 + 의존성 설치 완료")


In [2]:
import os
from getpass import getpass
from huggingface_hub import login

# (1) HF 토큰 — Write 권한 (나중에 GGUF repo 생성/업로드)
os.environ["HF_TOKEN"] = getpass("HF_TOKEN (Write 권한): ").strip()
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

# (2) OpenAI API Key — LLM-as-a-Judge
os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ").strip()

# (3) Langfuse Cloud — 03과 동일 Project
os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("LANGFUSE_PUBLIC_KEY (pk-lf-...): ").strip()
os.environ["LANGFUSE_SECRET_KEY"] = getpass("LANGFUSE_SECRET_KEY (sk-lf-...): ").strip()
os.environ["LANGFUSE_BASE_URL"]   = "https://jp.cloud.langfuse.com"
os.environ["LANGFUSE_HOST"]       = os.environ["LANGFUSE_BASE_URL"]

print("\n✅ 모든 인증 완료")

---
## Phase 2: HF Hub에서 SFT FP16 merged 모델 다운로드

02의 `push_to_hub_merged(..., save_method="merged_16bit")` 결과물을 그대로 받아옵니다.

확인 포인트:
- `safetensors` 여러 파일 (FP16 weight, ~4GB)
- `config.json` (모델 구조)
- `tokenizer.json` / `tokenizer_config.json` → 학습 때와 **같은 tokenizer**여야 입출력이 올바름
- 총 용량 ~4GB (FP16 merged가 맞는지 확인)

> ⚠️ 02에서 `merged_4bit`으로 저장했다면 다음 셀의 `convert_hf_to_gguf.py`가 실패합니다.


In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download, whoami

HF_USERNAME = whoami()["name"]
SFT_REPO    = f"{HF_USERNAME}/product-cs-qwen3.5-2b-sft"
LOCAL_DIR   = Path("sft-model-fp16")

print(f"다운로드 대상: {SFT_REPO}\n")
snapshot_download(
    repo_id=SFT_REPO,
    local_dir=str(LOCAL_DIR),
    token=True,  # private repo 대응
)

# 받은 파일 목록 + 총 용량
total_bytes = sum(p.stat().st_size for p in LOCAL_DIR.rglob("*") if p.is_file())
print(f"\n✅ 저장 위치: {LOCAL_DIR.resolve()}")
print(f"   총 용량: {total_bytes / 1e9:.2f} GB")
print(f"   파일 수: {sum(1 for _ in LOCAL_DIR.rglob('*') if _.is_file())}개")


---
## Phase 3: HF → GGUF 변환 (FP16 그대로)

이 단계는 **양자화가 아니라 포맷 변환**입니다 (14강 "그릇과 파일의 분리" 실증).

- `convert_hf_to_gguf.py`: HF safetensors 여러 개 → 단일 `.gguf` 한 파일로 패킹
- `--outtype f16`: 비트 수는 그대로 유지 (정밀도 손실 0)
- 결과 파일 크기는 원본과 거의 동일 (~4GB)

> 이 FP16 GGUF가 이후 양자화의 **입력 파일**이자, 03의 FP16 baseline과 직접 비교할 **포맷 변환 reference**입니다.


In [4]:
from pathlib import Path

GGUF_DIR = Path("gguf")
GGUF_DIR.mkdir(exist_ok=True)

FP16_GGUF = GGUF_DIR / "product-cs-fp16.gguf"

!python llama.cpp/convert_hf_to_gguf.py ./sft-model-fp16 \
    --outfile {FP16_GGUF} \
    --outtype f16

print(f"\n✅ GGUF FP16 생성: {FP16_GGUF}")
print(f"   크기: {FP16_GGUF.stat().st_size / 1e9:.2f} GB")


---
## Phase 4: 양자화 실행 (Q8_0 / Q4_K_M)

`llama-quantize` 바이너리로 같은 FP16 GGUF에서 두 버전을 찍어냅니다.

| 포맷 | 비트 | 설명 |
|------|------|------|
| Q8_0 | 8bit | 블록 단위, FP16 대비 **절반 크기**, 품질 거의 동일 |
| Q4_K_M | 4bit | K-block medium 변형, FP16 대비 **~1/3 크기**, 업계 표준 |

> 양자화는 입력 FP16 GGUF만 있으면 **CPU에서 1~2분**이면 끝납니다 (GPU 불필요).


In [5]:
from pathlib import Path

GGUF_DIR    = Path("gguf")
FP16_GGUF   = GGUF_DIR / "product-cs-fp16.gguf"
Q8_GGUF     = GGUF_DIR / "product-cs-q8_0.gguf"
Q4_GGUF     = GGUF_DIR / "product-cs-q4_k_m.gguf"

QUANTIZE_BIN = "./llama.cpp/build/bin/llama-quantize"

!{QUANTIZE_BIN} {FP16_GGUF} {Q8_GGUF} Q8_0
print()
!{QUANTIZE_BIN} {FP16_GGUF} {Q4_GGUF} Q4_K_M

print("\n" + "=" * 60)
print(f"{'포맷':<10}{'파일':<30}{'크기':>12}")
print("-" * 60)
for label, path in [("FP16", FP16_GGUF), ("Q8_0", Q8_GGUF), ("Q4_K_M", Q4_GGUF)]:
    size_gb = path.stat().st_size / 1e9
    print(f"{label:<10}{path.name:<30}{size_gb:>9.2f} GB")


---
## Phase 5: 파일 크기 비교 시각화

14강에서 배운 "7B × 0.5GB = 3.5GB" 공식이 실제 파일에서 어떻게 나타나는지 막대그래프로 확인합니다.

- 원본(FP16) 대비 압축률을 함께 표시
- **Q4_K_M이 왜 업계 표준(Ollama 기본값)인지**를 수치로 납득


In [6]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import subprocess
from pathlib import Path

# Colab에서 한글 폰트 자동 설치 (1회만)
nanum_path = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
if not nanum_path.exists():
    subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], check=False,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if nanum_path.exists():
    fm.fontManager.addfont(str(nanum_path))
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

labels = ["FP16", "Q8_0", "Q4_K_M"]
paths  = [FP16_GGUF, Q8_GGUF, Q4_GGUF]
sizes_gb = [p.stat().st_size / 1e9 for p in paths]
fp16_size = sizes_gb[0]
ratios    = [s / fp16_size * 100 for s in sizes_gb]

colors = ["#94a3b8", "#3b82f6", "#ef4444"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, sizes_gb, color=colors, edgecolor="black", linewidth=0.5)

for bar, size, ratio in zip(bars, sizes_gb, ratios):
    ax.annotate(f"{size:.2f} GB\n({ratio:.0f}%)",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 6), textcoords="offset points",
                ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("파일 크기 (GB)", fontsize=12)
ax.set_title("GGUF 정밀도별 파일 크기 비교 (원본 대비 %)", fontsize=13)
ax.set_ylim(0, max(sizes_gb) * 1.25)
ax.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

print("\n💡 Q4_K_M은 원본 대비 약 1/3 크기로 줄었지만, 품질은 거의 유지됩니다.")
print("   → 모바일/노트북 CPU에서 돌릴만한 수준!")


---
## Phase 6: 14강 핵심 포인트 — GPU 없이도 돌아간다

이 노트북은 전체가 **CPU 런타임**으로 동작합니다. 14강에서 배운 "GGUF + llama.cpp는 GPU 없이 동작한다"의 실증이에요.

- llama.cpp는 순수 C/C++ + SIMD 최적화 → CUDA 런타임 불필요
- `llama-cpp-python` CPU wheel만 사용 → `n_threads=4`로 멀티코어 CPU 추론
- 같은 GGUF 파일을 **맥북 · 라즈베리파이 · 일반 PC** 어디서든 동일하게 실행 가능

### 02·03에서 이어서 실행하는 경우

02·03은 GPU 런타임(T4)을 사용합니다. 04를 같은 세션에서 이어서 돌리려면 두 선택지가 있습니다:

1. **GPU 런타임을 유지한 채 04 실행** (권장 — Colab에서 가장 빠른 경로)
   - 04 코드는 모두 CPU 동작이라 GPU 위에서도 그대로 잘 돌아갑니다.
   - 02·03이 만든 `test.jsonl`이 `/content/` 또는 Drive에 그대로 있어서 추가 다운로드 없음.

2. **04를 새 CPU 런타임에서 시작**
   - Drive에 `test.jsonl`이 저장되어 있어야 함 (01·03 Phase 6에서 자동 저장됨).
   - SFT 모델은 HF Hub에서 다시 받습니다 (Phase 2 자동 수행).
   - 맥북·로컬 노트북에서 실습할 때도 같은 경로.

> ⚠️ Colab Runtime → Change runtime type을 누르면 VM이 교체되어 `/content/` 하위가 전부 소실됩니다. 02·03을 막 끝낸 직후라면 산출물이 Drive에 백업됐는지 확인 후 변경하세요.


---
## Phase 7: llama-cpp-python 설치 + 3버전 모두 로드

- `llama-cpp-python`: llama.cpp의 Python 바인딩 (자체 백엔드 포함, llama.cpp 빌드 바이너리와 독립적)
- CPU 전용 wheel이 기본 (GPU가 없어도 설치/실행 가능)
- `n_threads=4`: Colab 무료 vCPU 수에 맞춤
- `n_ctx=2048`: 학습 때 `max_seq_length`와 동일 (컨텍스트 일관성)

로드 후 **Q4_K_M으로 간단한 질문 1개**를 추론해 "GPU 없이 돌아간다"를 시각적으로 확인합니다.


In [7]:
# llama-cpp-python 설치 — Colab 환경에서 검증된 manylinux2014 wheel 직접 다운로드.
#
# 왜 직접 URL인가:
#   abetlen pip index(https://abetlen.github.io/llama-cpp-python/whl/cpu)는 cp312-cp312-linux_x86_64
#   wheel을 우선 매칭하는데, 이게 musllinux(Alpine) 기반이라 Colab의 일반 glibc 환경에서
#   "libc.musl-x86_64.so.1: cannot open shared object" 에러로 import 실패함.
#   → 검증된 manylinux2014 wheel URL을 직접 명시해 이 함정을 회피.
#
# 버전 0.3.21: Phase 1에서 clone한 llama.cpp 커밋 f535774와 정확히 같은 GGUF 포맷 사용
# (이 매칭이 깨지면 "Failed to load model from file" 에러 발생 → Phase 1 cell부터 재실행)
WHEEL_URL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.21/"
    "llama_cpp_python-0.3.21-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"
)

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llama-cpp-python"],
               stderr=subprocess.DEVNULL)

# 1차: 검증된 manylinux2014 wheel 직접 설치 (~30초)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", WHEEL_URL])

# 2차 fallback: URL이 사라진 경우 abetlen index로 시도
if r.returncode != 0:
    print("[install] ⚠️ 1차 직접 URL 실패 → abetlen index로 fallback")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cpu",
        "llama-cpp-python",
    ], check=True)
    try:
        import importlib, llama_cpp
        importlib.reload(llama_cpp)
        llama_cpp.llama_supports_gpu_offload()
    except Exception:
        print("[install] ⚠️ abetlen wheel도 musllinux 문제 → 컴파일 fallback (5~10분)")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "llama-cpp-python"],
                       stderr=subprocess.DEVNULL)
        subprocess.run([sys.executable, "-m", "pip", "install", "--prefer-binary",
                        "llama-cpp-python==0.3.21"], check=True)

# 빠른 검증: 공유 라이브러리 로드 + CPU-only wheel 확인
import importlib, llama_cpp
importlib.reload(llama_cpp)
print(f"✅ llama-cpp-python {llama_cpp.__version__} 로드 성공")
print(f"   GPU offload 지원: {llama_cpp.llama_supports_gpu_offload()}  (False면 CPU-only wheel ✅)")

from llama_cpp import Llama
from pathlib import Path
import gc

GGUF_DIR  = Path("gguf")
FP16_GGUF = GGUF_DIR / "product-cs-fp16.gguf"
Q8_GGUF   = GGUF_DIR / "product-cs-q8_0.gguf"
Q4_GGUF   = GGUF_DIR / "product-cs-q4_k_m.gguf"

# 사전 점검: 파일 존재 + 크기 + 메모리 정리
gc.collect()
print("\n--- GGUF 파일 사전 점검 ---")
for label, path in [("FP16-gguf", FP16_GGUF), ("Q8_0", Q8_GGUF), ("Q4_K_M", Q4_GGUF)]:
    if not path.exists():
        raise FileNotFoundError(
            f"❌ {label} GGUF 파일 없음: {path}\n"
            f"   Phase 3·4(cell-07, cell-09)를 다시 실행하세요."
        )
    size_gb = path.stat().st_size / 1e9
    if size_gb < 0.5:
        raise ValueError(
            f"❌ {label} 파일이 비정상적으로 작음 ({size_gb:.2f} GB) — 변환/양자화 실패 의심.\n"
            f"   Phase 3·4를 재실행하거나 파일을 삭제 후 재생성하세요."
        )
    print(f"  {label}: {size_gb:.2f} GB ✓")

def load_llm(path):
    # verbose=True: 모델 로드 실패 시 llama.cpp의 실제 에러 메시지가 stderr에 표시됨
    # ("unknown model architecture", "gguf version mismatch" 등 진단에 결정적)
    return Llama(
        model_path=str(path),
        n_threads=4,
        n_ctx=2048,
        verbose=True,
    )

print("\n모델 3종 로드 중... (각 30초~1분, Colab 무료 RAM 12.7GB 안에 모두 적재)")
llms = {}
for label, path in [("FP16-gguf", FP16_GGUF), ("Q8_0", Q8_GGUF), ("Q4_K_M", Q4_GGUF)]:
    print(f"\n[{label}] 로드 중...")
    llms[label] = load_llm(path)
    gc.collect()

print("\n✅ 3버전 모두 메모리에 로드 완료\n")

# --- 샘플 확인 ---
test_q = "주문한 PB 운동화가 사이즈가 작아요. 어떻게 해야 하나요?"
out = llms["Q4_K_M"].create_chat_completion(
    messages=[{"role": "user", "content": test_q}],
    max_tokens=192,
    temperature=0.0,
)
print("─" * 60)
print(f"[Q4_K_M 샘플 추론]\n질문: {test_q}")
print(f"답변: {out['choices'][0]['message']['content'][:300]}")
print("─" * 60)
print("\n💡 GPU 유무와 무관하게 답변이 나왔다면 14강 핵심 포인트 통과 ✅")


---
## Phase 8: Langfuse 초기화 (03 Project 재사용)

03에서 만든 **동일한 Langfuse Project**에 양자화 버전을 기록합니다. 그래야 대시보드에서 "학습 단계(Base/SFT) × 양자화 단계(FP16/Q8/Q4)" 조합을 한 화면에 비교할 수 있습니다.

- Trace `metadata.condition`: `SFT+FP16-gguf` / `SFT+Q8_0` / `SFT+Q4_K_M`
- Trace `tags`: `quantization`, `<quant_type>` → 대시보드 필터용
- `generation.end()`에 latency_ms 기록 → Langfuse UI의 Latency 컬럼에 자동 반영


In [8]:
import time
import langfuse as _lf_mod
from langfuse import get_client, propagate_attributes

langfuse = get_client()
print(f"✅ Langfuse 클라이언트 초기화 (SDK v{_lf_mod.__version__})")
print(f"   대시보드: {os.environ['LANGFUSE_BASE_URL']}")


CONDITION_LABEL = {
    "fp16-gguf": "SFT+FP16-gguf",
    "q8_0":      "SFT+Q8_0",
    "q4_k_m":    "SFT+Q4_K_M",
}

# CPU 추론 기준 추정 비용 (Colab CPU 런타임은 무료지만 표준화를 위해 GPU baseline과 동일 단가 사용)
ESTIMATED_CPU_COST_PER_HOUR_USD = 0.05


def generate_gguf_with_trace(llm, question: str, quant_type: str, max_tokens: int = 192):
    """
    llm        : llama_cpp.Llama 인스턴스
    quant_type : "fp16-gguf" | "q8_0" | "q4_k_m"
    반환       : {response, trace_id, observation_id, latency_ms, tokens_per_sec, ...}
    """
    cond_name = CONDITION_LABEL[quant_type]
    messages  = [{"role": "user", "content": question}]

    with propagate_attributes(tags=["quantization", "sft-eval", quant_type]):
        generation = langfuse.start_observation(
            as_type="generation",
            name="answer",
            model=f"qwen3.5-2b-sft-gguf-{quant_type}",
            input=messages,
            metadata={
                "condition": cond_name,
                "quantization": quant_type,
                "inference_engine": "llama.cpp",
                "question": question,
            },
        )

        t0 = time.perf_counter()
        res = llm.create_chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.0,
        )
        latency_s = time.perf_counter() - t0
        latency_ms = int(latency_s * 1000)

        text   = res["choices"][0]["message"]["content"]
        usage  = res.get("usage", {}) or {}
        input_tokens   = int(usage.get("prompt_tokens", 0))
        output_tokens  = int(usage.get("completion_tokens", 0))
        total_tokens   = input_tokens + output_tokens
        tokens_per_sec = output_tokens / latency_s if latency_s > 0 else 0.0

        est_cost = (latency_s / 3600.0) * ESTIMATED_CPU_COST_PER_HOUR_USD
        input_cost  = est_cost * (input_tokens  / total_tokens) if total_tokens else 0.0
        output_cost = est_cost * (output_tokens / total_tokens) if total_tokens else 0.0

        generation.update(
            output=text,
            usage_details={
                "input":  input_tokens,
                "output": output_tokens,
                "total":  total_tokens,
            },
            cost_details={
                "input":  round(input_cost,  8),
                "output": round(output_cost, 8),
                "total":  round(est_cost,    8),
            },
            metadata={
                "condition": cond_name,
                "quantization": quant_type,
                "inference_engine": "llama.cpp",
                "question": question,
                "latency_ms": latency_ms,
                "tokens_per_sec": round(tokens_per_sec, 2),
                "cost_method": "estimated_cpu_time",
            },
        )
        generation.end()

    return {
        "response": text,
        "trace_id": generation.trace_id,
        "observation_id": generation.id,
        "latency_ms": latency_ms,
        "tokens_per_sec": tokens_per_sec,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    }

print("\n✅ generate_gguf_with_trace 정의 완료")


---
## Phase 9: 평가 데이터 로드 (03과 동일 30개 재사용)

**공정한 비교의 전제**: 같은 질문으로 평가해야 03의 FP16 baseline과 숫자가 직접 비교됩니다.

- `random.seed(42)` + `random.sample(test_data, 30)` → 03과 **완전히 동일한 30개** 추출
- 양자화 실습은 품질 저하 측정이 목표이므로 OOD 5개는 생략 (Phase 13에서 수학 질문 따로 체험)


In [9]:
import json
import random
from pathlib import Path

random.seed(42)

LOCAL_PATH = Path("product-cs-sft-lab/sft-dataset/test.jsonl")
if LOCAL_PATH.exists():
    TEST_PATH = LOCAL_PATH
    print(f"📂 로컬 경로 사용: {TEST_PATH}")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    TEST_PATH = Path("/content/drive/MyDrive/product-cs-sft-lab/sft-dataset/test.jsonl")
    print(f"☁️  Drive 경로 사용: {TEST_PATH}")

assert TEST_PATH.exists(), f"test.jsonl을 찾을 수 없습니다: {TEST_PATH}"

test_data = [json.loads(l) for l in open(TEST_PATH, encoding="utf-8") if l.strip()]
N_EVAL = min(30, len(test_data))
eval_samples   = random.sample(test_data, N_EVAL)
eval_questions = [s["messages"][0]["content"] for s in eval_samples]

print(f"✅ 평가 질문 {len(eval_questions)}개 (03과 동일 seed=42 샘플)")
for q in eval_questions[:3]:
    print(f"  - {q[:60]}{'...' if len(q) > 60 else ''}")


---
## Phase 10: 3버전 답변 수집 + Trace 기록 + 속도 측정

총 `30 × 3 = 90개` 답변 생성. 각 답변마다:
- Langfuse Trace + Generation 자동 기록 (condition / quantization / latency / tokens)
- `tokens/sec` 측정 → 양자화가 **크기뿐 아니라 속도에도 영향**을 주는 것 확인 (14강 복습)

> ⏱️ CPU 추론이라 FP16은 느립니다(질문 당 30~60초). 30개 × 3버전 = 약 10~15분 소요.


In [10]:
from collections import defaultdict
from tqdm import tqdm

results   = []
speed_agg = defaultdict(list)

QUANT_ORDER = [
    ("fp16-gguf", "SFT+FP16-gguf"),
    ("q8_0",      "SFT+Q8_0"),
    ("q4_k_m",    "SFT+Q4_K_M"),
]
LLM_KEY = {"fp16-gguf": "FP16-gguf", "q8_0": "Q8_0", "q4_k_m": "Q4_K_M"}

for quant_type, cond_name in QUANT_ORDER:
    llm = llms[LLM_KEY[quant_type]]
    for q in tqdm(eval_questions, desc=f"[{cond_name}] 답변 수집"):
        try:
            rec = generate_gguf_with_trace(llm, q, quant_type)
        except Exception as e:
            print(f"⚠️  실패 ({cond_name}, {q[:30]}...): {e}")
            continue
        results.append({
            "question": q,
            "response": rec["response"],
            "condition": cond_name,
            "quantization": quant_type,
            "trace_id": rec["trace_id"],
            "observation_id": rec["observation_id"],
            "latency_ms": rec["latency_ms"],
            "tokens_per_sec": rec["tokens_per_sec"],
        })
        speed_agg[quant_type].append(rec["tokens_per_sec"])

langfuse.flush()

# --- 속도 요약 ---
print("\n" + "=" * 60)
print(f"{'양자화':<14}{'평균 tokens/sec':>18}{'평균 지연(s)':>16}")
print("-" * 60)
for quant_type, cond_name in QUANT_ORDER:
    tps_list = speed_agg[quant_type]
    if not tps_list:
        continue
    mean_tps = sum(tps_list) / len(tps_list)
    mean_lat = sum(r["latency_ms"] for r in results if r["quantization"] == quant_type) \
               / len(tps_list) / 1000
    print(f"{cond_name:<14}{mean_tps:>15.2f}     {mean_lat:>12.1f}")

print(f"\n✅ {len(results)}개 답변 + Trace 기록 완료")


---
## Phase 11: LLM-as-a-Judge 평가 + Langfuse Score 기록

**03과 동일한 Judge 프롬프트**를 재사용합니다 (판정 기준 일관성 보장). GPT-4o-mini로 90개 답변을 4차원(정확성·톤·구체성·환각)으로 채점하고 Langfuse Score로 첨부.

- 환각은 **낮을수록 좋음** (1=없음, 5=심각)
- 비용: ~$0.3 (90 × 4 Score ≈ 360건)


In [11]:
import json
from collections import defaultdict
from openai import OpenAI
from tqdm import tqdm

oai = OpenAI()

JUDGE_SYSTEM = (
    "당신은 쇼핑몰 고객 지원 품질 평가 전문가입니다. "
    "고객 질문에 대한 모델 답변을 1~5점 척도로 채점하고 JSON으로만 출력하세요."
)

def call_judge(question: str, response: str) -> dict:
    user_prompt = f"""[고객 질문] {question}

[모델 답변]
{response}

다음 4개 항목을 1~5점으로 평가하세요:
- 정확성 (5=완벽, 1=완전히 틀림)
- 톤 (5=상담원 어조 완벽, 1=무뚝뚝)
- 구체성 (5=단계별 명확, 1=모호)
- 환각 (1=없음, 5=심각하게 지어냄, ★ 낮을수록 좋음)

JSON으로만 출력 (다른 말 절대 금지):
{{"정확성": N, "톤": N, "구체성": N, "환각": N}}"""

    res = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(res.choices[0].message.content)


scores_by_cond = defaultdict(lambda: defaultdict(list))

for r in tqdm(results, desc="LLM-as-a-Judge 채점"):
    try:
        scores = call_judge(r["question"], r["response"])
    except Exception as e:
        print(f"⚠️  채점 실패 ({r['condition']}): {e}")
        continue

    for metric, value in scores.items():
        try:
            value_f = float(value)
        except (TypeError, ValueError):
            continue

        langfuse.create_score(
            trace_id=r["trace_id"],
            observation_id=r.get("observation_id"),
            name=metric,
            value=value_f,
            comment=r["condition"],
            data_type="NUMERIC",
        )
        scores_by_cond[r["condition"]][metric].append(value_f)

langfuse.flush()
print("\n✅ Judge 채점 + Langfuse Score 기록 완료")


---
## Phase 12: 양자화 손실 리포트 (클라이맥스)

FP16-gguf를 **양자화 baseline**으로 놓고 Q8_0 / Q4_K_M의 점수 차이를 계산합니다.

**해석 가이드**
- `Q4_K_M − FP16-gguf`가 **−0.3 이내**라면 일반 답변 태스크에서 양자화 영향 미미 (수용 가능)
- 환각 지표가 상승(= 나빠짐)한다면 양자화로 인한 오차 누적 신호
- 03의 `SFT+NoPrompt` (FP16 Transformers) 점수와 `SFT+FP16-gguf`가 비슷한지도 대시보드에서 교차 확인 → **포맷 변환은 무손실이어야 정상**


In [12]:
import statistics
import matplotlib.pyplot as plt

CONDS = ["SFT+FP16-gguf", "SFT+Q8_0", "SFT+Q4_K_M"]
METRICS = ["정확성", "톤", "구체성", "환각"]

# --- 평균 테이블 ---
mean = {}
print(f"{'조건':<18}{'정확성':>10}{'톤':>10}{'구체성':>10}{'환각(↓)':>12}")
print("-" * 60)
for c in CONDS:
    row = {m: statistics.mean(scores_by_cond[c][m]) if scores_by_cond[c][m] else 0.0
           for m in METRICS}
    mean[c] = row
    print(f"{c:<18}"
          f"{row['정확성']:>10.2f}"
          f"{row['톤']:>10.2f}"
          f"{row['구체성']:>10.2f}"
          f"{row['환각']:>12.2f}")

print("\n" + "=" * 60)
print("양자화 손실 = (FP16-gguf) - (Q_x)   (정확성/톤/구체성)")
print("환각 증가   = (Q_x) - (FP16-gguf)    (낮을수록 좋으므로 부호 반대)")
print("=" * 60)

# --- 델타 계산 ---
fp16 = mean["SFT+FP16-gguf"]
for c in ["SFT+Q8_0", "SFT+Q4_K_M"]:
    print(f"\n[{c} vs FP16-gguf]")
    for m in ["정확성", "톤", "구체성"]:
        delta = mean[c][m] - fp16[m]
        sign  = "+" if delta >= 0 else ""
        print(f"  {m:<6}: {sign}{delta:.2f}")
    halluc = mean[c]["환각"] - fp16["환각"]
    sign   = "+" if halluc >= 0 else ""
    print(f"  환각(↑나쁨): {sign}{halluc:.2f}")

# --- 라인 차트 ---
colors = {"SFT+FP16-gguf": "#94a3b8", "SFT+Q8_0": "#3b82f6", "SFT+Q4_K_M": "#ef4444"}

fig, ax = plt.subplots(figsize=(10, 5.5))
x = list(range(len(METRICS)))
for c in CONDS:
    y = [mean[c][m] for m in METRICS]
    ax.plot(x, y, marker="o", linewidth=2.5, markersize=10, label=c, color=colors[c])
    for i, v in enumerate(y):
        ax.annotate(f"{v:.2f}", (i, v), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=9, color=colors[c])

ax.set_xticks(x)
ax.set_xticklabels(METRICS, fontsize=12)
ax.set_ylabel("평균 점수 (1~5)", fontsize=12)
ax.set_title("양자화 정밀도별 Judge 점수 (환각은 낮을수록 좋음)", fontsize=13)
ax.legend(loc="lower left", fontsize=11)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_ylim(0, 5.2)
plt.tight_layout()
plt.show()


---
## Phase 13: (추가) 수학 / 다단계 추론 질문 체험

14강 **"양자화는 수학에 특히 취약 — 최대 69% 하락"** 경고를 실제로 눈으로 확인합니다.

- 2단계 이상 계산이 필요한 산수 5문제
- 세 버전에 동일 질문을 던져 정답 여부와 계산 과정의 일관성을 비교
- Q4_K_M에서 산수 오답이 눈에 띄게 늘면 → **양자화의 "오차 누적"이 실무에서 어떻게 나타나는지** 체감


In [13]:
math_questions = [
    "120 × 5 + 30은 얼마인가요? 계산 과정도 보여주세요.",
    "내가 사과 3개를 가지고 있고, 2개를 더 사고, 그중 1개를 먹었다면 몇 개 남았을까요?",
    "직사각형의 가로가 8cm, 세로가 12cm라면 넓이와 둘레는 각각 얼마인가요?",
    "한 상자에 연필이 24자루 있습니다. 5상자를 사서 3명이 똑같이 나눠 가지면 한 명당 몇 자루인가요?",
    "어떤 수의 2배에 7을 더했더니 25가 되었다면, 원래 수는 얼마인가요?",
]

def truncate(s, n=250):
    return s[:n] + ("..." if len(s) > n else "")

for q in math_questions:
    print("=" * 75)
    print(f"[질문] {q}")
    print("-" * 75)
    for quant_type, cond_name in QUANT_ORDER:
        out = llms[LLM_KEY[quant_type]].create_chat_completion(
            messages=[{"role": "user", "content": q}],
            max_tokens=256,
            temperature=0.0,
        )
        text = out["choices"][0]["message"]["content"]
        print(f"[{cond_name}]")
        print(f"  {truncate(text)}\n")

print("=" * 75)
print("💡 판단 팁:")
print("  • FP16-gguf / Q8_0는 같은 답을 내는 경우가 많다 (양자화 영향 작음)")
print("  • Q4_K_M에서만 계산 실수가 나타난다면 14강 '오차 누적'이 증명된 것")
print("  • 실무 시사점: 수학/코드/긴 추론 태스크는 Q8 이상 권장")


---
## Phase 14: Hugging Face Hub에 GGUF 3종 배포

같은 repo(`<user>/product-cs-qwen3.5-2b-gguf`)에 FP16 / Q8_0 / Q4_K_M 3종을 함께 올리고, 평가 결과 요약을 Model Card에 담습니다.

- repo 이름: `<user>/product-cs-qwen3.5-2b-gguf` (원본 `-sft`와 구분)
- 업로드 후 **Ollama / LM Studio에서 바로 pull 가능**한 상태가 됨


In [14]:
from huggingface_hub import HfApi, whoami, create_repo

HF_USERNAME = whoami()["name"]
GGUF_REPO   = f"{HF_USERNAME}/product-cs-qwen3.5-2b-gguf"

print(f"업로드 대상 repo: {GGUF_REPO}\n")

create_repo(repo_id=GGUF_REPO, repo_type="model", private=True, exist_ok=True)

# --- Model Card 생성 ---
sft_q8_delta_acc  = mean["SFT+Q8_0"]["정확성"]   - mean["SFT+FP16-gguf"]["정확성"]
sft_q4_delta_acc  = mean["SFT+Q4_K_M"]["정확성"] - mean["SFT+FP16-gguf"]["정확성"]

readme = f"""---
license: apache-2.0
base_model: {HF_USERNAME}/product-cs-qwen3.5-2b-sft
tags:
- gguf
- llama.cpp
- quantized
- qwen
- korean
- customer-support
---

# Product CS — Qwen3.5-2B SFT (GGUF)

Quantized GGUF versions of the product customer-support fine-tuned model.

## Base Model
- Architecture: Qwen3.5-2B
- Fine-tuning: QLoRA + SFT on product CS synthetic dataset
- Original FP16 model: [{HF_USERNAME}/product-cs-qwen3.5-2b-sft](https://huggingface.co/{HF_USERNAME}/product-cs-qwen3.5-2b-sft)

## Available Quantizations

| File | Size | Use Case |
|------|------|----------|
| product-cs-fp16.gguf   | {FP16_GGUF.stat().st_size/1e9:.2f} GB | Server / reference baseline |
| product-cs-q8_0.gguf   | {Q8_GGUF.stat().st_size/1e9:.2f} GB | Desktop, math-heavy tasks |
| product-cs-q4_k_m.gguf | {Q4_GGUF.stat().st_size/1e9:.2f} GB | Mobile, CPU, laptop ⭐ |

## Evaluation (LLM-as-a-Judge, GPT-4o-mini, N={N_EVAL})

| Metric | FP16-gguf | Q8_0 | Q4_K_M |
|--------|-----------|------|--------|
| 정확성 | {mean['SFT+FP16-gguf']['정확성']:.2f} | {mean['SFT+Q8_0']['정확성']:.2f} | {mean['SFT+Q4_K_M']['정확성']:.2f} |
| 톤     | {mean['SFT+FP16-gguf']['톤']:.2f} | {mean['SFT+Q8_0']['톤']:.2f} | {mean['SFT+Q4_K_M']['톤']:.2f} |
| 구체성 | {mean['SFT+FP16-gguf']['구체성']:.2f} | {mean['SFT+Q8_0']['구체성']:.2f} | {mean['SFT+Q4_K_M']['구체성']:.2f} |
| 환각(↓)| {mean['SFT+FP16-gguf']['환각']:.2f} | {mean['SFT+Q8_0']['환각']:.2f} | {mean['SFT+Q4_K_M']['환각']:.2f} |

Quantization accuracy delta vs FP16-gguf: Q8_0 {sft_q8_delta_acc:+.2f}, Q4_K_M {sft_q4_delta_acc:+.2f}

## Usage

### Ollama
```bash
ollama run hf.co/{GGUF_REPO}:Q4_K_M
```

### LM Studio
Search `{GGUF_REPO}` → download Q4_K_M variant.

### llama.cpp (Python)
```python
from llama_cpp import Llama
llm = Llama(model_path="product-cs-q4_k_m.gguf", n_ctx=2048, n_threads=4)
print(llm.create_chat_completion(
    messages=[{{"role": "user", "content": "화면이 깨져요"}}],
    max_tokens=256,
)["choices"][0]["message"]["content"])
```

## License
Apache-2.0 (inherits from base model)
"""

with open("gguf/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

# --- 파일 업로드 ---
api = HfApi()
for gguf_path in [FP16_GGUF, Q8_GGUF, Q4_GGUF]:
    print(f"⬆️  {gguf_path.name} 업로드 중... ({gguf_path.stat().st_size/1e9:.2f} GB)")
    api.upload_file(
        path_or_fileobj=str(gguf_path),
        path_in_repo=gguf_path.name,
        repo_id=GGUF_REPO,
        repo_type="model",
    )

api.upload_file(
    path_or_fileobj="gguf/README.md",
    path_in_repo="README.md",
    repo_id=GGUF_REPO,
    repo_type="model",
)

print(f"\n✅ 배포 완료: https://huggingface.co/{GGUF_REPO}")


---

## 완료

| 결과물 | 위치 |
|--------|------|
| GGUF 3종 파일 | 로컬 `gguf/` + `<user>/product-cs-qwen3.5-2b-gguf` (HF Hub) |
| 90개 Trace + 360개 Score | 03과 동일 Langfuse Project (tag `quantization`) |
| 양자화 손실 정량값 | Phase 12 출력 + Model Card |
| 수학 민감도 체험 | Phase 13 출력 (14강 "오차 누적" 실증) |

> **다음**: `05_gradio-lab-plan.md` — 배포한 GGUF를 Gradio로 감싸서 공개 데모 만들기.
